In [ ]:
# CELDA 1: SETUP + MODELOS

import sys
import subprocess
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

print("⏳ Configurando entorno...\n")

packages = [
    "open-clip-torch==2.23.0",
    "transformers==4.35.2",
    "scikit-learn",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    print(f"✅ {pkg}")

# Importar módulos
from utils.model_factory import create_model, list_available_models
from utils.embeddings import get_all_patch_embeddings_from_image
from utils.multi_step_clustering import (
    run_block_clustering_on_embeddings,
    decidir_fondo_vs_tejido,
    decidir_nucleos_vs_citoplasma
)
from utils.visualization import visualizar_clusters_basicos
from utils.coco_loader import list_coco_images, load_image_and_segmentations_from_coco

print("\nModelos disponibles:")
for name, desc in list_available_models().items():
    print(f"  - {name}: {desc}")

# Cargar ambos modelos
print("\n🔬 Cargando BiomedCLIP...")
model_biomedclip = create_model('biomedclip')

print("🔬 Cargando UNI...")
model_uni = create_model('uni')

print("\n✅ Setup completado")

In [ ]:
# CELDA 2: CARGAR IMAGEN Y GROUND TRUTH

JSON_PATH = r"C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train\_annotations.coco.json"
IMAGES_DIR = r"C:\Users\mngra\projects\AI\Pap\PAP_DATA\CCEDD\CCEDD-UTN-10imgs\train"

print("📂 Explorando dataset...")
images_list = list_coco_images(JSON_PATH)
print(f"Imágenes disponibles: {len(images_list)}")

IMAGE_INDEX = 3
CATEGORY_IDS = [4, 5]  # kernel y kernel-not-confirmed

print(f"\n📷 Cargando imagen {IMAGE_INDEX}...")
img_original, segmentations, fname, (H_orig, W_orig) = load_image_and_segmentations_from_coco(
    JSON_PATH, IMAGES_DIR,
    image_id=images_list[IMAGE_INDEX]["id"],
    block_size=224,
    category_ids=CATEGORY_IDS
)

# Resize
TARGET_SIZE = (1376, 1020)
scale = min(TARGET_SIZE[0] / img_original.shape[1], TARGET_SIZE[1] / img_original.shape[0])
new_w = int(img_original.shape[1] * scale)
new_h = int(img_original.shape[0] * scale)
img_resized = cv2.resize(img_original, (new_w, new_h), interpolation=cv2.INTER_AREA)

# Escalar segmentations
segmentations_scaled = []
for seg in segmentations:
    scaled_coords = [[int(x * scale), int(y * scale)] for x, y in seg['polygon_coords']]
    segmentations_scaled.append({
        'category_id': seg['category_id'],
        'polygon_coords': scaled_coords,
        'annotation_id': seg.get('annotation_id', None)
    })
segmentations = segmentations_scaled

# Convertir a grises
if img_resized.ndim == 3:
    img = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
else:
    img = img_resized

H, W = img.shape[:2]
print(f"✅ {fname}")
print(f"   Dimensiones: {W}x{H}")
print(f"   Núcleos GT: {len(segmentations)}")

# Visualizar
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(img, cmap='gray')
for i, seg in enumerate(segmentations):
    polygon = np.array(seg['polygon_coords'], dtype=np.int32)
    if len(polygon) >= 3:
        ax.plot(np.append(polygon[:, 0], polygon[0, 0]),
                np.append(polygon[:, 1], polygon[0, 1]),
                'b-', linewidth=2, label='GT' if i == 0 else '')
ax.set_title(f"Imagen + GT ({len(segmentations)} núcleos)", fontweight='bold')
ax.axis('off')
if len(segmentations) > 0:
    ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# CELDA DIAGNÓSTICO: DIBUJAR SOLO GROUND TRUTH (TODAS LAS CATEGORÍAS)

import json
from matplotlib.patches import Patch

print("\n" + "="*70)
print("DIAGNÓSTICO GT: TODAS LAS CATEGORÍAS")
print("="*70)

# 1) Mapa de categorías desde JSON COCO
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)
cat_id_to_name = {int(c['id']): c['name'] for c in coco_data.get('categories', [])}

# 2) Cargar segmentaciones SIN filtro de categorías
img_gt_all, segmentations_all, fname_all, _ = load_image_and_segmentations_from_coco(
    JSON_PATH,
    IMAGES_DIR,
    image_id=images_list[IMAGE_INDEX]["id"],
    block_size=224,
    category_ids=None
 )

print(f"Imagen: {fname_all}")
print(f"Total anotaciones (todas las clases): {len(segmentations_all)}")

# 3) Reescalar igual que en celda 2 para comparar visualmente
scale_all = min(TARGET_SIZE[0] / img_gt_all.shape[1], TARGET_SIZE[1] / img_gt_all.shape[0])
new_w_all = int(img_gt_all.shape[1] * scale_all)
new_h_all = int(img_gt_all.shape[0] * scale_all)
img_gt_all_resized = cv2.resize(img_gt_all, (new_w_all, new_h_all), interpolation=cv2.INTER_AREA)

segmentations_all_scaled = []
for seg in segmentations_all:
    scaled_coords = [[int(x * scale_all), int(y * scale_all)] for x, y in seg['polygon_coords']]
    segmentations_all_scaled.append({
        'category_id': int(seg['category_id']),
        'polygon_coords': scaled_coords,
        'annotation_id': seg.get('annotation_id', None)
    })

# 4) Colores por categoría
cat_color = {
    0: '#808080',   # cells-Xkl0
    1: '#1f77b4',   # cell
    2: '#17becf',   # cell-not-confirmed
    3: '#9467bd',   # isolated-kernel
    4: '#2ca02c',   # kernel
    5: '#d62728',   # kernel-not-confirmed
}

# 5) Dibujar solo GT
fig, ax = plt.subplots(1, 1, figsize=(14, 10))
ax.imshow(img_gt_all_resized)

counts = {}
for seg in segmentations_all_scaled:
    cat = int(seg['category_id'])
    counts[cat] = counts.get(cat, 0) + 1
    poly = np.array(seg['polygon_coords'], dtype=np.int32)
    if len(poly) >= 3:
        color = cat_color.get(cat, '#ffff00')
        ax.plot(np.append(poly[:, 0], poly[0, 0]),
                np.append(poly[:, 1], poly[0, 1]),
                color=color, linewidth=1.8, alpha=0.9)

legend_items = []
for cat_id in sorted(counts.keys()):
    name = cat_id_to_name.get(cat_id, f'cat-{cat_id}')
    legend_items.append(Patch(facecolor=cat_color.get(cat_id, '#ffff00'), edgecolor='black',
                              label=f"{cat_id}: {name} (n={counts[cat_id]})"))

if len(legend_items) > 0:
    ax.legend(handles=legend_items, loc='upper right', fontsize=9, framealpha=0.95)

ax.set_title('Ground Truth (todas las categorías, solo anotaciones)', fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

print("\nConteo por categoría en esta imagen:")
for cat_id in sorted(counts.keys()):
    print(f"  - {cat_id}: {cat_id_to_name.get(cat_id, 'unknown')} -> {counts[cat_id]}")

In [ ]:
print("\n🔬 Extrayendo embeddings UNI...")
patch_data_uni = get_all_patch_embeddings_from_image(
    img_resized, model_uni,
    preprocess=None,
    tile_size=128
)

In [ ]:
print("\n🔬 Extrayendo embeddings BIOMEDCLIP...")
patch_data_biomedclip = get_all_patch_embeddings_from_image(
    img_resized, model_biomedclip,
    preprocess=None,
    tile_size=128
)

In [ ]:
# CELDA 6: MÁSCARAS POR CLASE (núcleo / citoplasma / fondo)

nucleus_cats = {3, 4, 5}

cytoplasm_cats = {1, 2}



mask_nucleus = np.zeros((H, W), dtype=np.uint8)

mask_cytoplasm = np.zeros((H, W), dtype=np.uint8)



for seg in segmentations_all_scaled:

    poly = np.array(seg['polygon_coords'], dtype=np.int32)

    if poly.shape[0] < 3:

        continue

    cat = int(seg['category_id'])

    if cat in nucleus_cats:

        cv2.fillPoly(mask_nucleus, [poly], 1)

    elif cat in cytoplasm_cats:

        cv2.fillPoly(mask_cytoplasm, [poly], 1)



mask_tissue = ((mask_nucleus | mask_cytoplasm) > 0).astype(np.uint8)

mask_background = (mask_tissue == 0).astype(np.uint8)



fig, axs = plt.subplots(1, 3, figsize=(15, 5))

axs[0].imshow(img, cmap='gray')

axs[0].imshow(mask_nucleus, cmap='Reds', alpha=0.4)

axs[0].set_title('Núcleos')

axs[0].axis('off')



axs[1].imshow(img, cmap='gray')

axs[1].imshow(mask_cytoplasm, cmap='Oranges', alpha=0.4)

axs[1].set_title('Citoplasma')

axs[1].axis('off')



axs[2].imshow(img, cmap='gray')

axs[2].imshow(mask_background, cmap='Blues', alpha=0.35)

axs[2].set_title('Fondo')

axs[2].axis('off')



plt.tight_layout()

plt.show()

In [ ]:
# CELDA 7: Etiquetar patches y estadísticas básicas por grupo

from collections import defaultdict



def annotate_patches(patches, modality):

    enriched = []

    for p in patches:

        x1, y1, x2, y2 = map(int, p['position'])

        roi_nuc = mask_nucleus[y1:y2, x1:x2]

        roi_cyto = mask_cytoplasm[y1:y2, x1:x2]

        area = max(1, (y2 - y1) * (x2 - x1))

        frac_nuc = float(roi_nuc.sum()) / area

        frac_cyto = float(roi_cyto.sum()) / area

        frac_bg = max(0.0, 1.0 - frac_nuc - frac_cyto)

        if frac_nuc >= max(frac_cyto, frac_bg):

            label = "nucleo"

        elif frac_cyto >= frac_bg:

            label = "citoplasma"

        else:

            label = "fondo"

        enriched.append({

            **p,

            'label': label,

            'frac_nuc': frac_nuc,

            'frac_cyto': frac_cyto,

            'frac_bg': frac_bg,

            'intensity_mean': float(img[y1:y2, x1:x2].mean()) if y2 > y1 and x2 > x1 else 0.0,

            'modality': modality

        })

    return enriched



patches_uni = annotate_patches(patch_data_uni, modality="uni")

patches_biomed = annotate_patches(patch_data_biomedclip, modality="biomedclip")



def summarize_groups(patches):

    summary = {}

    total = len(patches)

    for group in ["nucleo", "citoplasma", "fondo"]:

        sub = [p for p in patches if p['label'] == group]

        if not sub:

            summary[group] = dict(count=0, frac=0.0, intensity_mean=np.nan, intensity_std=np.nan)

            continue

        intens = np.array([p['intensity_mean'] for p in sub], dtype=float)

        summary[group] = dict(

            count=len(sub),

            frac=len(sub) / total,

            intensity_mean=float(intens.mean()),

            intensity_std=float(intens.std())

        )

    return summary



print("📊 UNI:", summarize_groups(patches_uni))

print("📊 BIOMEDCLIP:", summarize_groups(patches_biomed))

In [ ]:
# CELDA 8: PCA 2D de embeddings por grupo



def project_and_plot(patches, title):

    valid = [p for p in patches if p.get('embedding') is not None]

    if len(valid) < 10:

        print("Muy pocos patches para PCA")

        return

    X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])

    labels = [p['label'] for p in valid]

    pca = PCA(n_components=2, random_state=42)

    X2 = pca.fit_transform(X)

    colors = dict(nucleo='red', citoplasma='darkorange', fondo='gray')

    fig, ax = plt.subplots(figsize=(7, 6))

    for g in sorted(set(labels)):

        idx = [i for i, l in enumerate(labels) if l == g]

        ax.scatter(X2[idx, 0], X2[idx, 1], s=14, c=colors.get(g, 'blue'), label=g, alpha=0.7)

    ax.set_title(f"{title} (PCA 2D) - var expl {pca.explained_variance_ratio_.sum():.2f}")

    ax.set_xlabel("PC1")

    ax.set_ylabel("PC2")

    ax.legend()

    plt.tight_layout()

    plt.show()



project_and_plot(patches_uni, "UNI")

project_and_plot(patches_biomed, "BiomedCLIP")

In [ ]:
# CELDA 9: Clustering no supervisado y comparación (píxeles vs embeddings)

from collections import Counter



def eval_unsupervised(patches, n_clusters=3):

    valid = [p for p in patches if p.get('embedding') is not None]

    if len(valid) < n_clusters * 2:

        print("No hay suficientes patches para clustering")

        return None

    X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])

    true_labels = [p['label'] for p in valid]

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)

    pred_clusters = km.fit_predict(X)

    mapping = {}

    for cid in range(n_clusters):

        idx = [i for i, c in enumerate(pred_clusters) if c == cid]

        maj = Counter(true_labels[i] for i in idx).most_common(1)

        mapping[cid] = maj[0][0] if maj else "desconocido"

    pred_labels = [mapping[c] for c in pred_clusters]

    true_nuc = np.array([l == "nucleo" for l in true_labels], dtype=bool)

    pred_nuc = np.array([l == "nucleo" for l in pred_labels], dtype=bool)

    tp = int((true_nuc & pred_nuc).sum())

    fp = int((~true_nuc & pred_nuc).sum())

    fn = int((true_nuc & ~pred_nuc).sum())

    precision = tp / (tp + fp + 1e-9)

    recall = tp / (tp + fn + 1e-9)

    return dict(mapping=mapping, counts=dict(tp=tp, fp=fp, fn=fn), precision=precision, recall=recall)



def eval_pixel_baseline(patches, n_clusters=3):

    X = np.array([[p['intensity_mean']] for p in patches], dtype=float)

    true_labels = [p['label'] for p in patches]

    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)

    pred_clusters = km.fit_predict(X)

    mapping = {}

    for cid in range(n_clusters):

        idx = [i for i, c in enumerate(pred_clusters) if c == cid]

        maj = Counter(true_labels[i] for i in idx).most_common(1)

        mapping[cid] = maj[0][0] if maj else "desconocido"

    pred_labels = [mapping[c] for c in pred_clusters]

    true_nuc = np.array([l == "nucleo" for l in true_labels], dtype=bool)

    pred_nuc = np.array([l == "nucleo" for l in pred_labels], dtype=bool)

    tp = int((true_nuc & pred_nuc).sum())

    fp = int((~true_nuc & pred_nuc).sum())

    fn = int((true_nuc & ~pred_nuc).sum())

    precision = tp / (tp + fp + 1e-9)

    recall = tp / (tp + fn + 1e-9)

    return dict(mapping=mapping, precision=precision, recall=recall)



res_uni = eval_unsupervised(patches_uni)

res_biomed = eval_unsupervised(patches_biomed)

res_pixel = eval_pixel_baseline(patches_uni)



print("UNI clustering:", res_uni)

print("BiomedCLIP clustering:", res_biomed)

print("Baseline píxeles:", res_pixel)

In [ ]:
# CELDA 10: Cobertura de máscaras e intensidad promedio

area_total = H * W

pix_nuc = int(mask_nucleus.sum())

pix_cyto = int(mask_cytoplasm.sum())

pix_bg = int(mask_background.sum())



def pct(x):

    return 100.0 * x / max(1, area_total)



stats_masks = {

    "pixeles": dict(nucleo=pix_nuc, citoplasma=pix_cyto, fondo=pix_bg),

    "porcentaje": dict(nucleo=pct(pix_nuc), citoplasma=pct(pix_cyto), fondo=pct(pix_bg)),

    "intensidad_media": dict(

        nucleo=float(img[mask_nucleus == 1].mean()) if pix_nuc > 0 else np.nan,

        citoplasma=float(img[mask_cytoplasm == 1].mean()) if pix_cyto > 0 else np.nan,

        fondo=float(img[mask_background == 1].mean()) if pix_bg > 0 else np.nan,

    ),

}



print("Cobertura (%):", stats_masks["porcentaje"])

print("Pixeles:", stats_masks["pixeles"])

print("Intensidad media:", stats_masks["intensidad_media"])



fig, ax = plt.subplots(1, 2, figsize=(10, 4))

ax[0].bar(["nucleo", "citoplasma", "fondo"], [stats_masks["porcentaje"][k] for k in ["nucleo", "citoplasma", "fondo"]], color=["red", "orange", "gray"])

ax[0].set_ylabel("% imagen")

ax[0].set_title("Cobertura")



ax[1].bar(["nucleo", "citoplasma", "fondo"], [stats_masks["intensidad_media"][k] for k in ["nucleo", "citoplasma", "fondo"]], color=["red", "orange", "gray"])

ax[1].set_ylabel("Intensidad promedio (0-255)")

ax[1].set_title("Brillo por clase")



plt.tight_layout()

plt.show()

In [ ]:
# CELDA 11: Histogramas de intensidad por clase

bins = np.linspace(0, 255, 50)

vals = {

    "nucleo": img[mask_nucleus == 1].ravel(),

    "citoplasma": img[mask_cytoplasm == 1].ravel(),

    "fondo": img[mask_background == 1].ravel(),

}



fig, ax = plt.subplots(figsize=(8, 5))

for k, c in [("nucleo", "red"), ("citoplasma", "darkorange"), ("fondo", "gray")]:

    if vals[k].size > 0:

        ax.hist(vals[k], bins=bins, alpha=0.45, label=k, color=c, density=True)

ax.set_title("Distribución de intensidad por clase")

ax.set_xlabel("Valor de pixel (0-255)")

ax.set_ylabel("Densidad")

ax.legend()

plt.tight_layout()

plt.show()

In [ ]:
# CELDA 12: Morfología de anotaciones (área, aspecto, circularidad)

def poly_metrics(seg):

    poly = np.array(seg['polygon_coords'], dtype=np.int32)

    if poly.shape[0] < 3:

        return None

    area = float(cv2.contourArea(poly))

    peri = float(cv2.arcLength(poly, True))

    x, y, w, h = cv2.boundingRect(poly)

    aspect = w / max(1.0, h)

    circularity = 4.0 * np.pi * area / max(1e-6, peri * peri)

    return dict(area=area, aspect=aspect, circularity=circularity)



metrics = {"nucleo": [], "citoplasma": []}

for seg in segmentations_all_scaled:

    cat = int(seg['category_id'])

    m = poly_metrics(seg)

    if m is None:

        continue

    if cat in nucleus_cats:

        metrics["nucleo"].append(m)

    elif cat in cytoplasm_cats:

        metrics["citoplasma"].append(m)



def summarize_metric(key):

    for cls in ["nucleo", "citoplasma"]:

        vals = np.array([d[key] for d in metrics[cls]], dtype=float)

        if vals.size == 0:

            print(f"{cls} sin datos para {key}")

            continue

        print(f"{cls} {key}: mean={vals.mean():.1f} std={vals.std():.1f} p50={np.percentile(vals, 50):.1f} p90={np.percentile(vals, 90):.1f}")



for k in ["area", "aspect", "circularity"]:

    summarize_metric(k)



fig, axs = plt.subplots(1, 3, figsize=(15, 4))

for i, (k, title) in enumerate(zip(["area", "aspect", "circularity"], ["Área px2", "Aspecto w/h", "Circularidad"])):

    for cls, color in [("nucleo", "red"), ("citoplasma", "darkorange")]:

        vals = np.array([d[k] for d in metrics[cls]], dtype=float)

        if vals.size > 0:

            axs[i].hist(vals, bins=30, alpha=0.5, color=color, label=cls)

    axs[i].set_title(title)

    axs[i].legend()

plt.tight_layout()

plt.show()

In [ ]:
# CELDA 13: t-SNE de embeddings (UNI y BiomedCLIP)

from sklearn.manifold import TSNE



def run_tsne(patches, title, max_samples=1500):

    valid = [p for p in patches if p.get('embedding') is not None]

    if len(valid) < 50:

        print("Pocos patches para t-SNE")

        return

    if len(valid) > max_samples:

        idx = np.random.choice(len(valid), max_samples, replace=False)

        valid = [valid[i] for i in idx]

    X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])

    labels = [p['label'] for p in valid]

    perp = max(5, min(30, len(valid) // 3))

    tsne = TSNE(n_components=2, perplexity=perp, learning_rate="auto", init="pca", random_state=42)

    X2 = tsne.fit_transform(X)

    colors = dict(nucleo='red', citoplasma='darkorange', fondo='gray')

    fig, ax = plt.subplots(figsize=(7, 6))

    for g in sorted(set(labels)):

        idx = [i for i, l in enumerate(labels) if l == g]

        ax.scatter(X2[idx, 0], X2[idx, 1], s=8, c=colors.get(g, 'blue'), alpha=0.65, label=g)

    ax.set_title(f"{title} t-SNE (n={len(valid)})")

    ax.axis('off')

    ax.legend()

    plt.tight_layout()

    plt.show()



run_tsne(patches_uni, "UNI")

run_tsne(patches_biomed, "BiomedCLIP")

In [ ]:
# CELDA 14: Barrido de k y métricas de clustering (silhouette, Davies-Bouldin, pureza)

from sklearn.metrics import silhouette_score, davies_bouldin_score



def k_sweep(patches, ks=range(2, 7)):

    valid = [p for p in patches if p.get('embedding') is not None]

    if len(valid) < max(ks) * 3:

        print("Pocos patches para barrido de k")

        return []

    X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])

    labels = [p['label'] for p in valid]

    results = []

    for k in ks:

        km = KMeans(n_clusters=k, random_state=42, n_init=10)

        pred = km.fit_predict(X)

        sil = silhouette_score(X, pred) if len(set(pred)) > 1 else np.nan

        db = davies_bouldin_score(X, pred) if len(set(pred)) > 1 else np.nan

        pur = 0.0

        for cid in set(pred):

            idx = [i for i, c in enumerate(pred) if c == cid]

            maj = Counter([labels[i] for i in idx]).most_common(1)

            pur += (len(idx) / len(pred)) * (maj[0][1] / len(idx))

        results.append(dict(k=k, silhouette=sil, db=db, pureza=pur))

    return results



res_uni_k = k_sweep(patches_uni)

res_bio_k = k_sweep(patches_biomed)



def plot_k(res, title):

    if not res:

        print("Sin resultados para", title)

        return

    ks = [r['k'] for r in res]

    fig, ax = plt.subplots(1, 3, figsize=(14, 4))

    ax[0].plot(ks, [r['silhouette'] for r in res], marker='o')

    ax[0].set_title("Silhouette")

    ax[1].plot(ks, [r['db'] for r in res], marker='o')

    ax[1].set_title("Davies-Bouldin (bajo=mejor)")

    ax[2].plot(ks, [r['pureza'] for r in res], marker='o')

    ax[2].set_title("Pureza vs GT parches")

    for a in ax:

        a.set_xlabel("k")

        a.grid(True, alpha=0.3)

    fig.suptitle(title)

    plt.tight_layout()

    plt.show()



plot_k(res_uni_k, "UNI barrido k")

plot_k(res_bio_k, "BiomedCLIP barrido k")

In [ ]:
# CELDA 15: Estabilidad de clustering (repeticiones KMeans)

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score



def clustering_runs(patches, k=3, runs=8):

    valid = [p for p in patches if p.get('embedding') is not None]

    if len(valid) < k * 3:

        print("Muy pocos patches para estabilidad")

        return None, None

    X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])

    labels_runs = []

    for seed in range(runs):

        km = KMeans(n_clusters=k, random_state=100 + seed, n_init=10)

        labels_runs.append(km.fit_predict(X))

    ari_mat = np.zeros((runs, runs))

    nmi_mat = np.zeros((runs, runs))

    for i in range(runs):

        for j in range(runs):

            ari_mat[i, j] = adjusted_rand_score(labels_runs[i], labels_runs[j])

            nmi_mat[i, j] = normalized_mutual_info_score(labels_runs[i], labels_runs[j])

    return ari_mat, nmi_mat



ari_uni, nmi_uni = clustering_runs(patches_uni)

ari_bio, nmi_bio = clustering_runs(patches_biomed)



def plot_heat(mat, title, vmin=0, vmax=1):

    if mat is None:

        return

    fig, ax = plt.subplots(figsize=(5, 4))

    im = ax.imshow(mat, vmin=vmin, vmax=vmax, cmap='viridis')

    ax.set_title(title)

    ax.set_xlabel("run")

    ax.set_ylabel("run")

    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()

    plt.show()



plot_heat(ari_uni, "ARI UNI (estabilidad)")

plot_heat(nmi_uni, "NMI UNI (estabilidad)")

plot_heat(ari_bio, "ARI BiomedCLIP (estabilidad)")

plot_heat(nmi_bio, "NMI BiomedCLIP (estabilidad)")



if ari_uni is not None:

    print("Promedio ARI UNI:", float(ari_uni[np.triu_indices_from(ari_uni, 1)].mean()))

if ari_bio is not None:

    print("Promedio ARI BiomedCLIP:", float(ari_bio[np.triu_indices_from(ari_bio, 1)].mean()))

In [ ]:
# CELDA 16: Separación núcleos vs resto (ARI/NMI) para embeddings y píxeles

def ari_nmi(patches, k=3, use_embedding=True):

    if use_embedding:

        valid = [p for p in patches if p.get('embedding') is not None]

        if len(valid) < k * 3:

            return None

        X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])

        labels = [p['label'] for p in valid]

    else:

        valid = patches

        if len(valid) < k * 3:

            return None

        X = np.array([[p['intensity_mean']] for p in valid], dtype=float)

        labels = [p['label'] for p in valid]

    km = KMeans(n_clusters=k, random_state=42, n_init=10)

    pred = km.fit_predict(X)

    return dict(

        ari=adjusted_rand_score(labels, pred),

        nmi=normalized_mutual_info_score(labels, pred)

    )



scores = {

    "uni": ari_nmi(patches_uni, use_embedding=True),

    "biomedclip": ari_nmi(patches_biomed, use_embedding=True),

    "pixeles": ari_nmi(patches_uni, use_embedding=False)

}

print(scores)

In [ ]:
# CELDA 17: Máscara de núcleos predicha desde clustering (k=3)

def cluster_to_mask(patches, modality_name):

    valid = [p for p in patches if p.get('embedding') is not None]

    if len(valid) < 30:

        print("Muy pocos patches")

        return None

    X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])

    labels_gt = [p['label'] for p in valid]

    km = KMeans(n_clusters=3, random_state=42, n_init=10)

    pred = km.fit_predict(X)

    mapping = {}

    for cid in range(3):

        idx = [i for i, c in enumerate(pred) if c == cid]

        maj = Counter(labels_gt[i] for i in idx).most_common(1)

        mapping[cid] = maj[0][0] if maj else "desconocido"

    mask_pred = np.zeros_like(mask_nucleus, dtype=np.uint8)

    for p, cid in zip(valid, pred):

        if mapping[cid] != "nucleo":

            continue

        x1, y1, x2, y2 = map(int, p['position'])

        mask_pred[y1:y2, x1:x2] = 1

    tp = int((mask_pred & mask_nucleus).sum())

    fp = int((mask_pred & (mask_nucleus == 0)).sum())

    fn = int(((mask_nucleus == 1) & (mask_pred == 0)).sum())

    precision = tp / (tp + fp + 1e-9)

    recall = tp / (tp + fn + 1e-9)

    iou = tp / (tp + fp + fn + 1e-9)

    print(f"{modality_name}: mapping={mapping} precision={precision:.3f} recall={recall:.3f} IoU={iou:.3f}")

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))

    ax[0].imshow(img, cmap='gray'); ax[0].set_title('Imagen'); ax[0].axis('off')

    ax[1].imshow(mask_nucleus, cmap='Reds'); ax[1].set_title('GT núcleos'); ax[1].axis('off')

    ax[2].imshow(mask_pred, cmap='Reds'); ax[2].set_title(f'Pred núcleos {modality_name}'); ax[2].axis('off')

    plt.tight_layout(); plt.show()

    return dict(mapping=mapping, precision=precision, recall=recall, iou=iou, mask=mask_pred)



res_mask_uni = cluster_to_mask(patches_uni, "UNI")

res_mask_bio = cluster_to_mask(patches_biomed, "BiomedCLIP")

In [ ]:
# CELDA 18: VISUALIZACIÓN Y ANÁLISIS DE SCORES ARI/NMI

import pandas as pd

scores_data = {
    "uni": {"ari": 0.4945, "nmi": 0.4231},
    "biomedclip": {"ari": 0.2851, "nmi": 0.3534},
    "pixeles": {"ari": 0.3817, "nmi": 0.3919}
}

df_scores = pd.DataFrame(scores_data).T
df_scores['promedio'] = df_scores[['ari', 'nmi']].mean(axis=1)
print("\n" + "="*70)
print("COMPARATIVA ARI/NMI (k=3 clusters)")
print("="*70)
print(df_scores.round(4))
print("\n✅ VEREDICTO GENERAL:")
print(f"  🥇 UNI es {((df_scores.loc['uni', 'promedio'] - df_scores.loc['pixeles', 'promedio']) / df_scores.loc['pixeles', 'promedio'] * 100):.1f}% mejor que píxeles")
print(f"  ❌ BiomedCLIP es {((df_scores.loc['pixeles', 'promedio'] - df_scores.loc['biomedclip', 'promedio']) / df_scores.loc['pixeles', 'promedio'] * 100):.1f}% peor que píxeles")

# Visualizar comparativa
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

# ARI
ax[0].bar(df_scores.index, df_scores['ari'], color=['#2ca02c', '#d62728', '#1f77b4'], alpha=0.7, edgecolor='black', linewidth=2)
ax[0].set_ylabel('ARI Score', fontsize=12, fontweight='bold')
ax[0].set_title('ARI: Clustering vs Ground Truth\n(+1 perfecto, 0 aleatorio)', fontweight='bold')
ax[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
ax[0].set_ylim(-0.1, 0.6)
for i, v in enumerate(df_scores['ari']):
    ax[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax[0].grid(axis='y', alpha=0.3)

# NMI
ax[1].bar(df_scores.index, df_scores['nmi'], color=['#2ca02c', '#d62728', '#1f77b4'], alpha=0.7, edgecolor='black', linewidth=2)
ax[1].set_ylabel('NMI Score', fontsize=12, fontweight='bold')
ax[1].set_title('NMI: Información Compartida\n(1 perfecto, 0 nada)', fontweight='bold')
ax[1].set_ylim(0, 0.5)
for i, v in enumerate(df_scores['nmi']):
    ax[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Tabla de ganancia/pérdida
print("\n" + "="*70)
print("GANANCIA RELATIVA vs BASELINE (píxeles)")
print("="*70)
pixel_ari = df_scores.loc['pixeles', 'ari']
pixel_nmi = df_scores.loc['pixeles', 'nmi']

for modelo in ['uni', 'biomedclip']:
    gain_ari = ((df_scores.loc[modelo, 'ari'] - pixel_ari) / pixel_ari * 100)
    gain_nmi = ((df_scores.loc[modelo, 'nmi'] - pixel_nmi) / pixel_nmi * 100)
    promedio = (gain_ari + gain_nmi) / 2
    print(f"\n{modelo.upper()}:")
    print(f"  ARI: {gain_ari:+.1f}%")
    print(f"  NMI: {gain_nmi:+.1f}%")
    print(f"  Promedio: {promedio:+.1f}%")


In [ ]:
# CELDA 19 (CORREGIDA): DIAGNÓSTICO - ¿Por qué precision/recall = 0?

print("\n" + "="*70)
print("DIAGNÓSTICO: DEBUGGING MÁSCARA PREDICHA")
print("="*70)

def cluster_to_mask_debug(patches, modality_name):
    valid = [p for p in patches if p.get('embedding') is not None]
    print(f"\n{modality_name}:")
    print(f"  Total patches: {len(valid)}")
    
    if len(valid) < 30:
        print("  ❌ Muy pocos patches")
        return None

    X = np.stack([np.asarray(p['embedding']).ravel() for p in valid])
    labels_gt = [p['label'] for p in valid]
    
    # Conteo GT
    from collections import Counter
    cnt = Counter(labels_gt)
    print(f"  Distribución GT: {dict(cnt)}")

    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    pred = km.fit_predict(X)
    pred_cnt = Counter(pred)
    print(f"  Clusters predichos: {dict(pred_cnt)}")

    # Mapping
    mapping = {}
    for cid in range(3):
        idx = [i for i, c in enumerate(pred) if c == cid]
        maj = Counter(labels_gt[i] for i in idx).most_common(1)
        mapping[cid] = maj[0][0] if maj else "desconocido"
        comp = Counter([labels_gt[i] for i in idx])
        print(f"    Cluster {cid} ({len(idx)} patches): {dict(comp)} -> asignado a '{mapping[cid]}'")

    # Crear máscara
    mask_pred = np.zeros_like(mask_nucleus, dtype=np.uint8)
    nucleos_patches = 0
    
    for p, cid in zip(valid, pred):
        if mapping[cid] != "nucleo":
            continue
        nucleos_patches += 1
        x1, y1, x2, y2 = map(int, p['position'])
        x1, x2 = max(0, x1), min(W, x2)
        y1, y2 = max(0, y1), min(H, y2)
        mask_pred[y1:y2, x1:x2] = 1

    print(f"  Patches etiquetados como núcleo: {nucleos_patches}")
    print(f"  Píxeles predichos como núcleo: {int(mask_pred.sum())}")
    print(f"  Píxeles GT núcleo: {int(mask_nucleus.sum())}")

    # Métricas
    tp = int((mask_pred & mask_nucleus).sum())
    fp = int((mask_pred & (mask_nucleus == 0)).sum())
    fn = int(((mask_nucleus == 1) & (mask_pred == 0)).sum())
    tn = int(((mask_nucleus == 0) & (mask_pred == 0)).sum())

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    iou = tp / (tp + fp + fn + 1e-9)

    print(f"  TP={tp}, FP={fp}, FN={fn}, TN={tn}")
    print(f"  Precision={precision:.3f}, Recall={recall:.3f}, F1={f1:.3f}, IoU={iou:.3f}")

    # Visualizar
    fig, axs = plt.subplots(2, 3, figsize=(15, 10))
    
    axs[0, 0].imshow(img, cmap='gray')
    axs[0, 0].set_title('Imagen original')
    axs[0, 0].axis('off')
    
    axs[0, 1].imshow(mask_nucleus, cmap='Reds', alpha=0.7)
    axs[0, 1].set_title(f'GT núcleos ({int(mask_nucleus.sum())} px)')
    axs[0, 1].axis('off')
    
    axs[0, 2].imshow(mask_pred, cmap='Blues', alpha=0.7)
    axs[0, 2].set_title(f'Pred núcleos ({int(mask_pred.sum())} px)')
    axs[0, 2].axis('off')
    
    # Overlay
    overlay = np.zeros((H, W, 3), dtype=np.uint8)
    overlay[mask_nucleus == 1, 0] = 255  # Rojo = GT
    overlay[mask_pred == 1, 2] = 255     # Azul = Pred
    overlay[(mask_nucleus & mask_pred) == 1, :] = [255, 0, 255]  # Magenta = TP
    
    axs[1, 0].imshow(overlay)
    axs[1, 0].set_title(f'GT (rojo) vs Pred (azul) - TP (magenta)')
    axs[1, 0].axis('off')
    
    # Histograma de distancias a clusters (CORREGIDO)
    distances = km.transform(X)
    # Encontrar índice del cluster asignado a "nucleo"
    nucleo_cluster_id = [cid for cid, label in mapping.items() if label == "nucleo"]
    if nucleo_cluster_id:
        nuc_id = nucleo_cluster_id[0]
        axs[1, 1].hist(distances[:, nuc_id], bins=30, alpha=0.6, color='green', edgecolor='black')
        axs[1, 1].set_title(f'Distancias al cluster {nuc_id} (asignado a núcleo)')
        axs[1, 1].set_xlabel('Distancia Euclidiana')
        axs[1, 1].set_ylabel('Frecuencia')
    else:
        axs[1, 1].text(0.5, 0.5, 'Sin cluster de núcleo', ha='center', va='center')
        axs[1, 1].set_title('⚠️ Sin núcleos predichos')
    
    # Métricas text
    axs[1, 2].axis('off')
    metrics_text = f"""
RESULTADOS {modality_name.upper()}
    
Mapping: {mapping}

TP={tp}  FP={fp}
FN={fn}  TN={tn}

Precision: {precision:.3f}
Recall:    {recall:.3f}
F1-Score:  {f1:.3f}
IoU:       {iou:.3f}
    """
    axs[1, 2].text(0.1, 0.5, metrics_text, fontsize=11, family='monospace',
                   verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

    return dict(mapping=mapping, precision=precision, recall=recall, f1=f1, iou=iou, mask=mask_pred)

res_debug_uni = cluster_to_mask_debug(patches_uni, "UNI")
res_debug_bio = cluster_to_mask_debug(patches_biomed, "BiomedCLIP")